In [6]:
#Import all the python packages
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [7]:
!pip install aif360


In [8]:
#imported Aif360 packages
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import BinaryLabelDatasetMetric

In [9]:
data = pd.read_csv('/content/data_summary_amd.csv')

In [10]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   filename       10000 non-null  object 
 1   age            10000 non-null  float64
 2   gender         10000 non-null  object 
 3   race           10000 non-null  object 
 4   ethnicity      10000 non-null  object 
 5   language       10000 non-null  object 
 6   maritalstatus  10000 non-null  object 
 7   amd            10000 non-null  object 
 8   use            10000 non-null  object 
dtypes: float64(1), object(8)
memory usage: 703.3+ KB


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [12]:
data = data.drop(columns=['filename'])


In [13]:
# Step 3: Inspect the structure of the dataset
print(data.head())
print(data.info())
print(f"Unique values in 'amd' column before encoding: {data['amd'].unique()}")
print(f"Unique values in 'use' column: {data['use'].unique()}")


# Handle missing values
data = data.dropna()

# Encode categorical variables
label_encoders = {}
for column in data.columns:
    if data[column].dtype == 'object' and column != 'amd':
        le = LabelEncoder()
        data[column] = le.fit_transform(data[column])
        label_encoders[column] = le

# Encode the target variable 'amd'
le_amd = LabelEncoder()
data['amd'] = le_amd.fit_transform(data['amd'])

# Print the unique values of the target variable after encoding
print(f"Unique values in 'amd' column after encoding: {data['amd'].unique()}")

# Create binary target variable
data['amd_binary'] = np.where(data['amd'] == 3, 1, 0)
print(f"Unique values in 'amd_binary' column: {data['amd_binary'].unique()}")

# Step 5: Define target and sensitive features
target = 'amd_binary'
sensitive_feature = 'race'

# Print unique values of 'use' column to check for correct categories
print(f"Unique values in 'use' column: {data['use'].unique()}")

# Split the data into training, validation, and test sets based on the 'use' column
train_data = data[data['use'] == 0]  # Assuming 'training' is encoded as 0
val_data = data[data['use'] == 1]  # Assuming 'validation' is encoded as 1
test_data = data[data['use'] == 2]  # Assuming 'test' is encoded as 2

# Check for empty dataframes
print(f"Training data shape: {train_data.shape}")
print(f"Validation data shape: {val_data.shape}")
print(f"Test data shape: {test_data.shape}")

if train_data.empty or val_data.empty or test_data.empty:
    raise ValueError("One of the data splits (train, validation, test) is empty. Check 'use' column encoding.")

X_train = train_data.drop(columns=[target, sensitive_feature, 'use', 'amd'])
y_train = train_data[target]
X_val = val_data.drop(columns=[target, sensitive_feature, 'use', 'amd'])
y_val = val_data[target]
X_test = test_data.drop(columns=[target, sensitive_feature, 'use', 'amd'])
y_test = test_data[target]

# Step 6: Train a logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 7: Evaluate the model's performance
y_pred = model.predict(X_val)
print(f"Accuracy: {accuracy_score(y_val, y_pred)}")
print(classification_report(y_val, y_pred))

# Check for correct labels
favorable_label = 1
unfavorable_label = 0
print(f"Favorable label: {favorable_label}, Unfavorable label: {unfavorable_label}")

# Step 8: Measure disparate impact before mitigation
test_bld = BinaryLabelDataset(df=pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True), test_data[sensitive_feature].reset_index(drop=True)], axis=1),
                              label_names=[target],
                              protected_attribute_names=[sensitive_feature],
                              favorable_label=favorable_label,
                              unfavorable_label=unfavorable_label)

metric_test_bld = BinaryLabelDatasetMetric(test_bld, privileged_groups=[{sensitive_feature: 1}],
                                           unprivileged_groups=[{sensitive_feature: 0}])
print(f"Disparate impact before mitigation: {metric_test_bld.disparate_impact()}")

# Step 9: Apply the Reweighing algorithm to mitigate bias
RW = Reweighing(unprivileged_groups=[{sensitive_feature: 0}], privileged_groups=[{sensitive_feature: 1}])
train_bld = BinaryLabelDataset(df=pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True), train_data[sensitive_feature].reset_index(drop=True)], axis=1),
                               label_names=[target],
                               protected_attribute_names=[sensitive_feature],
                               favorable_label=favorable_label,
                               unfavorable_label=unfavorable_label)
train_bld_transf = RW.fit_transform(train_bld)

# Step 10: Retrain the logistic regression model
model_transf = LogisticRegression(max_iter=1000)
X_train_transf = train_bld_transf.features
y_train_transf = train_bld_transf.labels.ravel()
model_transf.fit(X_train_transf, y_train_transf)

# Step 11: Measure the disparate impact after mitigation
test_bld_transf = BinaryLabelDataset(df=pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True), test_data[sensitive_feature].reset_index(drop=True)], axis=1),
                                     label_names=[target],
                                     protected_attribute_names=[sensitive_feature],
                                     favorable_label=favorable_label,
                                     unfavorable_label=unfavorable_label)
metric_test_bld_transf = BinaryLabelDatasetMetric(test_bld_transf, privileged_groups=[{sensitive_feature: 1}],
                                                  unprivileged_groups=[{sensitive_feature: 0}])
print(f"Disparate impact after mitigation: {metric_test_bld_transf.disparate_impact()}")

# Step 12: Evaluate the model's performance after mitigation
X_test_transf = test_bld_transf.features
y_pred_transf = model_transf.predict(X_test_transf)
print(f"Accuracy after mitigation: {accuracy_score(y_test, y_pred_transf)}")

     age  gender   race     ethnicity language         maritalstatus  \
0  71.93  female  white  non-hispanic  english  married or partnered   
1  62.65    male  white      hispanic  english                single   
2  62.59  female  white  non-hispanic  english  married or partnered   
3  90.08  female  white  non-hispanic  english               widowed   
4  71.89  female  white  non-hispanic  english               unknown   

                amd       use  
0          late amd  training  
1         early amd  training  
2  intermediate amd  training  
3          late amd  training  
4         early amd  training  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   age            10000 non-null  float64
 1   gender         10000 non-null  object 
 2   race           10000 non-null  object 
 3   ethnicity      10000 non-null  object 
 4  

In [14]:
# Investigate the distribution of the binary target variable across protected groups
print(data.groupby(['race', 'amd_binary']).size())


race  amd_binary
0     0              158
      1              548
1     0               83
      1             1088
2     0             3328
      1             4795
dtype: int64
